# EXP-008 dev-small 확인: 안건 항목 법률명 조건

코퍼스 안건에서 뽑은 법률명 사전의 크기·예시와, single-doc dev 질의의 법률명 조건 수, dev-small 순위가 기준과 같은지 본다. multi-doc(G3) 질의는 보지 않는다(규칙을 G3로 다듬지 않는다, slices/11). 판정에 쓰지 않는다.

In [6]:
import collections
import json
import random
from pathlib import Path

from rag.conditions import agenda_laws
from rag.index import load_cfg

In [7]:
ROOT = Path.cwd().parent
CFG = load_cfg(['+exp=exp008'])
MIN_CHARS = CFG.conditions.law_min_chars
CORPUS = ROOT / CFG.paths.corpus
COND = ROOT / 'data' / 'conditions' / 'summary_q-law.jsonl'
RUNS = ROOT / 'data' / 'runs'

## 코퍼스 안건의 법률명

In [8]:
conf_laws = {}
n_docs = 0
with CORPUS.open(encoding='utf-8') as f:
    for doc in map(json.loads, f):
        n_docs += 1
        conf_laws[doc['conference_number']] = agenda_laws(doc['agenda'], MIN_CHARS)
freq = collections.Counter(k for v in conf_laws.values() for k in v)
print('회의', len(conf_laws), '/ 법률명이 있는 회의', sum(bool(v) for v in conf_laws.values()))
print('고유 법률명', len(freq))
print(freq.most_common(10))
random.seed(0)
print(random.sample(sorted(freq), 15))

회의 9662 / 법률명이 있는 회의 2446
고유 법률명 1539
[('소득세법', 182), ('법인세법', 159), ('국세기본법', 154), ('조세특례제한법', 149), ('부가가치세법', 131), ('관세법', 113), ('상속세및증여세법', 113), ('공직선거법', 109), ('주세법', 106), ('국세징수법', 106)]
['소년법', '식품산업진흥법', '검역법', '도서관법', '자격기본법', '위헌결정에따른학교용지부담금환급등에관한특별법', '수산자원관리법', '방위산업발전법', '원양산업발전법', '산지관리법', '중증장애인연금법', '농산물품질관리법', '인적자원개발기본법', '국방군사시설이전특별회계법', '문화재보호기금법']


## single-doc dev 질의의 법률명 조건

In [9]:
rows = [json.loads(line) for line in COND.open(encoding='utf-8')]
with_law = [r for r in rows if 'law' in r['conditions']]
print('dev 질의', len(rows), '/ law 추출', len(with_law),
      '/ law 포함 필드 2개 이상', sum(len(r['conditions']) >= 2 for r in with_law))
print(collections.Counter(v for r in with_law for v in r['conditions']['law']).most_common(10))

dev 질의 3016 / law 추출 58 / law 포함 필드 2개 이상 3
[('국가보안법', 4), ('국회법', 4), ('고용보험법', 3), ('근로기준법', 3), ('형사소송법', 2), ('개인정보보호법', 2), ('초중등교육법', 2), ('연구개발특구의육성에관한특별법', 1), ('법인세법', 1), ('조세특례제한법', 1)]


## dev-small 순위

In [10]:
base = (RUNS / 'kure-v1-fixed-512-64-hybrid-cond-filter-dev-small.csv').read_bytes()
cand = (RUNS / 'kure-v1-fixed-512-64-hybrid-cond-filter-law-dev-small.csv').read_bytes()
print('기준과 같음', base == cand)

기준과 같음 True
